# CCFULL-ROM parameter analysis

This notebook analyzes one CCFULL-ROM batch output. To analyze another
reaction or run, change only `INPUT_FILE` in the first code cell. The reaction
label and parameter coordinates are read automatically from the output files.

The sampled points are ranked with

$$
\chi^2(\boldsymbol\theta)=\sum_i
\left[\frac{\sigma_{\rm ROM}(E_i;\boldsymbol\theta)
-\sigma_{\rm exp}(E_i)}{\Delta\sigma_i}\right]^2.
$$

The supplied lower or upper experimental uncertainty is selected according
to the residual direction. Each figure is contained in one code cell.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

def find_project_root(start=Path.cwd()):
    for folder in (start, *start.parents):
        if (folder / "pyproject.toml").is_file():
            return folder
    raise FileNotFoundError("Run the notebook from inside CCFULL_ROM-new")

ROOT = find_project_root()
OUTPUT_ROOT = ROOT / "emulator" / "outputs"

# Change only this line. A run directory or its cross_sections.csv is accepted.
INPUT_FILE = "20261104_001409_553140_16O_154Sm_Basis_300/cross_sections.csv"
input_path = Path(INPUT_FILE).expanduser()
if not input_path.is_absolute():
    input_path = OUTPUT_ROOT / input_path
if input_path.is_dir():
    input_path = input_path / "cross_sections.csv"
DATA_FILE = input_path.resolve()
RUN_DIR = DATA_FILE.parent
SUMMARY_FILE = RUN_DIR / "run_summary.json"

if not DATA_FILE.is_file():
    raise FileNotFoundError(f"Input table not found: {DATA_FILE}")

summary = json.loads(SUMMARY_FILE.read_text(encoding="utf-8")) if SUMMARY_FILE.is_file() else {}
reaction_system = summary.get("reaction_system", RUN_DIR.name)
data = pd.read_csv(DATA_FILE)

summary_parameters = list(summary.get("parameter_domain", {}))
fallback_parameters = ["beta2", "beta3", "beta4", "beta6", "gamma", "v0", "r0", "a0"]
PARAMETERS = [name for name in summary_parameters if name in data.columns]
if not PARAMETERS:
    PARAMETERS = [name for name in fallback_parameters if name in data.columns]
if not PARAMETERS:
    raise ValueError("No emulator parameter columns were found")

required = {
    "point_index", "energy_cm_mev", "emulator_cross_section_mb",
    "experimental_cross_section_mb",
    "experimental_uncertainty_lower_mb",
    "experimental_uncertainty_upper_mb",
}
missing = sorted(required.difference(data.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")

fit = data.dropna(subset=[
    "experimental_cross_section_mb",
    "experimental_uncertainty_lower_mb",
    "experimental_uncertainty_upper_mb",
]).copy()
if fit.empty:
    raise ValueError("The selected output contains no experimental data for parameter fitting")

residual = fit["emulator_cross_section_mb"] - fit["experimental_cross_section_mb"]
fit["sigma_used_mb"] = np.where(
    residual >= 0.0,
    fit["experimental_uncertainty_upper_mb"],
    fit["experimental_uncertainty_lower_mb"],
)
if (fit["sigma_used_mb"] <= 0.0).any():
    raise ValueError("Experimental uncertainties must be positive")
fit["pull"] = residual / fit["sigma_used_mb"]

scores = fit.groupby("point_index").agg(
    chi2=("pull", lambda values: float(np.square(values).sum())),
    fitted_energies=("pull", "size"),
)
scores["chi2_per_point"] = scores["chi2"] / scores["fitted_energies"]
scores["log10_chi2_per_point"] = np.log10(scores["chi2_per_point"])

parameter_table = data.groupby("point_index")[PARAMETERS].first()
scan = parameter_table.join(scores).reset_index()
active_parameters = [name for name in PARAMETERS if scan[name].nunique() > 1]
fixed_parameters = {
    name: float(scan[name].iloc[0])
    for name in PARAMETERS if scan[name].nunique() == 1
}
if not active_parameters:
    raise ValueError("The selected output has no scanned parameter")

best = scan.loc[scan["chi2"].idxmin()]
best_index = int(best["point_index"])

print(f"Reaction system: {reaction_system}")
loaded_name = (DATA_FILE.relative_to(ROOT.resolve()) if DATA_FILE.is_relative_to(ROOT.resolve())
               else DATA_FILE)
print(f"Loaded: {loaded_name}")
print(f"Parameter points: {scan.shape[0]}")
print(f"Experimental energies per point: {int(best['fitted_energies'])}")
print(f"Active parameters: {active_parameters}")
print(f"Fixed parameters: {fixed_parameters}")
print(f"Best point: {best_index}; chi2/N = {best['chi2_per_point']:.6g}")
best[PARAMETERS + ["chi2", "chi2_per_point"]].to_frame("value")


Reaction system: A=16, Z=8 + A=154, Z=62
Loaded: emulator/outputs/20261104_001409_553140_16O_154Sm_Basis_300/cross_sections.csv
Parameter points: 100
Experimental energies per point: 39
Active parameters: ['beta2', 'beta4', 'v0', 'r0', 'a0']
Fixed parameters: {'beta6': 0.0}
Best point: 63; chi2/N = 14.2314


,value
beta2,0.320074
beta4,0.054747
beta6,0.000000
v0,180.172039
r0,0.947273
a0,1.022471
chi2,555.026544
chi2_per_point,14.231450


## Best parameters and local uncertainty

Around the best sampled point, the ROM response at each experimental energy is
fitted locally as a linear function of the active, range-normalized parameters,

$$
\sigma_k(\mathbf z)\approx c_k+\sum_a J_{ka}z_a.
$$

The local Gauss--Newton/Fisher covariance is

$$
\mathrm{Cov}(\mathbf z)=
\left[J^T WJ\right]^+,
\qquad W_{kk}=1/(\Delta\sigma_k)^2.
$$

`statistical_1sigma` uses the quoted experimental uncertainties directly.
`reported_1sigma` is additionally multiplied by the Birge factor
$\sqrt{\max[1,\chi^2/(N_E-N_p)]}$ when the residual scatter is larger than
the quoted errors. Fixed parameters are listed but are not assigned a fitted
uncertainty. These are local approximate uncertainties; they do not include
ROM truncation error or uncertainty from sparse parameter sampling.


In [2]:
# Local response fit around the best sampled point
n_active = len(active_parameters)
theta = scan[active_parameters].to_numpy(dtype=float)
theta_best = best[active_parameters].to_numpy(dtype=float)
theta_scale = np.ptp(theta, axis=0)
normalized_offset = (theta - theta_best) / theta_scale
distance = np.linalg.norm(normalized_offset, axis=1)

neighbor_count = min(len(scan), max(60, 10 * n_active))
neighbor_rows = np.argsort(distance)[:neighbor_count]
bandwidth = max(float(distance[neighbor_rows[-1]]), np.finfo(float).eps)
local_weight = np.exp(-0.5 * (distance[neighbor_rows] / bandwidth) ** 2)

curve_table = (
    data.pivot_table(
        index="point_index",
        columns="energy_cm_mev",
        values="emulator_cross_section_mb",
        aggfunc="first",
    )
    .sort_index(axis=1)
    .reindex(scan["point_index"])
)
design = np.column_stack([
    np.ones(neighbor_count),
    normalized_offset[neighbor_rows],
])
sqrt_weight = np.sqrt(local_weight)[:, None]
local_coefficients = np.linalg.lstsq(
    design * sqrt_weight,
    curve_table.to_numpy()[neighbor_rows] * sqrt_weight,
    rcond=None,
)[0]
jacobian = local_coefficients[1:].T

experiment_by_energy = (
    data[[
        "energy_cm_mev", "experimental_cross_section_mb",
        "experimental_uncertainty_lower_mb",
        "experimental_uncertainty_upper_mb",
    ]]
    .drop_duplicates("energy_cm_mev")
    .set_index("energy_cm_mev")
    .reindex(curve_table.columns)
)
best_prediction = curve_table.loc[best_index].to_numpy(dtype=float)
observed = experiment_by_energy["experimental_cross_section_mb"].to_numpy(dtype=float)
residual_at_best = best_prediction - observed
uncertainty_at_best = np.where(
    residual_at_best >= 0.0,
    experiment_by_energy["experimental_uncertainty_upper_mb"],
    experiment_by_energy["experimental_uncertainty_lower_mb"],
).astype(float)

usable_energy = (
    np.isfinite(observed)
    & np.isfinite(best_prediction)
    & np.isfinite(uncertainty_at_best)
    & (uncertainty_at_best > 0.0)
    & np.all(np.isfinite(jacobian), axis=1)
)
weighted_jacobian = jacobian[usable_energy] / uncertainty_at_best[usable_energy, None]
fisher = weighted_jacobian.T @ weighted_jacobian
fisher_rank = int(np.linalg.matrix_rank(fisher))
fisher_condition = float(np.linalg.cond(fisher))
covariance_normalized = np.linalg.pinv(fisher, rcond=1e-12)
statistical_1sigma = theta_scale * np.sqrt(np.maximum(np.diag(covariance_normalized), 0.0))

dof = max(int(usable_energy.sum()) - n_active, 1)
birge_factor = float(np.sqrt(max(1.0, best["chi2"] / dof)))
reported_1sigma = statistical_1sigma * birge_factor

uncertainty_table = pd.DataFrame(index=PARAMETERS)
uncertainty_table.index.name = "parameter"
uncertainty_table["best_sampled"] = best[PARAMETERS].astype(float)
uncertainty_table["statistical_1sigma"] = np.nan
uncertainty_table["reported_1sigma"] = np.nan
uncertainty_table["status"] = "fixed (not inferred)"
uncertainty_table.loc[active_parameters, "statistical_1sigma"] = statistical_1sigma
uncertainty_table.loc[active_parameters, "reported_1sigma"] = reported_1sigma
uncertainty_table.loc[active_parameters, "status"] = "active"
uncertainty_table["best_plus_minus_reported_1sigma"] = [
    (f"{row.best_sampled:.6g} ± {row.reported_1sigma:.3g}"
     if row.status == "active" else f"{row.best_sampled:.6g} (fixed)")
    for row in uncertainty_table.itertuples()
]

print(f"Local fit neighbors: {neighbor_count}")
print(f"Fisher rank: {fisher_rank}/{n_active}; condition number: {fisher_condition:.3g}")
print(f"Birge scale factor: {birge_factor:.4g}")
if fisher_rank < n_active:
    print("WARNING: the local Fisher matrix is rank deficient; uncertainties are not identifiable.")
uncertainty_table


Local fit neighbors: 60
Fisher rank: 5/5; condition number: 6.68e+03
Birge scale factor: 4.04


,best_sampled,statistical_1sigma,reported_1sigma,status,best_plus_minus_reported_1sigma
parameter,,,,,
beta2,0.320074,0.003969,0.016035,active,0.320074 ± 0.016
beta4,0.054747,0.002968,0.011994,active,0.0547469 ± 0.012
beta6,0.000000,NaN,NaN,fixed (not inferred),0 (fixed)
v0,180.172039,2.036881,8.229684,active,180.172 ± 8.23
r0,0.947273,0.002367,0.009564,active,0.947273 ± 0.00956
a0,1.022471,0.005676,0.022931,active,1.02247 ± 0.0229


## Parameter profiles

In [3]:
profile = scan.melt(
    id_vars=["point_index", "chi2_per_point", "log10_chi2_per_point"],
    value_vars=active_parameters,
    var_name="parameter",
    value_name="value",
)
fig = px.scatter(
    profile,
    x="value",
    y="log10_chi2_per_point",
    color="log10_chi2_per_point",
    facet_col="parameter",
    facet_col_wrap=3,
    color_continuous_scale="Viridis_r",
    labels={
        "value": "Parameter value",
        "log10_chi2_per_point": "log10(chi2/N)",
    },
    title=f"{reaction_system}: parameter profiles across sampled emulator points",
)
fig.update_traces(marker={"size": 5, "opacity": 0.75})
fig.update_xaxes(matches=None)
fig.update_yaxes(matches=None)
fig.update_layout(height=650)
fig.show()

## Parameter correlations among the best sampled points

In [4]:
top_fraction = 0.10
top_count = max(20, int(np.ceil(top_fraction * len(scan))))
top = scan.nsmallest(top_count, "chi2")

fig = px.scatter_matrix(
    top,
    dimensions=active_parameters,
    color="log10_chi2_per_point",
    color_continuous_scale="Viridis_r",
    labels={"log10_chi2_per_point": "log10(chi2/N)"},
    title=(f"{reaction_system}: parameter correlations; "
           f"best {top_count} of {len(scan)} sampled points"),
)
fig.update_traces(
    diagonal_visible=False,
    showupperhalf=False,
    marker={"size": 5, "opacity": 0.75},
)
fig.update_layout(height=850)
fig.show()

## Fusion excitation function at the best sampled point

In [5]:
best_curve = data.loc[data["point_index"] == best_index].sort_values("energy_cm_mev")
experiment = (
    data[[
        "energy_cm_mev", "experimental_cross_section_mb",
        "experimental_uncertainty_lower_mb",
        "experimental_uncertainty_upper_mb",
    ]]
    .drop_duplicates("energy_cm_mev")
    .sort_values("energy_cm_mev")
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=experiment["energy_cm_mev"],
    y=experiment["experimental_cross_section_mb"],
    error_y={
        "type": "data",
        "array": experiment["experimental_uncertainty_upper_mb"],
        "arrayminus": experiment["experimental_uncertainty_lower_mb"],
        "symmetric": False,
        "visible": True,
    },
    mode="markers",
    marker={"color": "black", "size": 7, "symbol": "circle-open"},
    name="Experiment",
))
fig.add_trace(go.Scatter(
    x=best_curve["energy_cm_mev"],
    y=best_curve["emulator_cross_section_mb"],
    mode="lines+markers",
    line={"color": "#0072B2", "width": 2},
    marker={"size": 5},
    name="CCFULL-ROM",
))
if "exact_numerov_cross_section_mb" in best_curve.columns:
    fig.add_trace(go.Scatter(
        x=best_curve["energy_cm_mev"],
        y=best_curve["exact_numerov_cross_section_mb"],
        mode="lines",
        line={"color": "#D55E00", "width": 2, "dash": "dash"},
        name="Exact Numerov",
    ))

parameter_text = ", ".join(f"{name}={best[name]:.6g}" for name in PARAMETERS)
fig.update_layout(
    title=(f"{reaction_system}: best sampled point {best_index}"
           f"<br><sup>fusion excitation function; "
           f"{parameter_text}; chi2/N={best['chi2_per_point']:.4g}</sup>"),
    xaxis_title="Centre-of-mass energy (MeV)",
    yaxis_title="Fusion cross section (mb)",
    yaxis_type="log",
    template="plotly_white",
    height=600,
    legend={"x": 0.02, "y": 0.98},
)
fig.show()

## Fusion barrier distribution at the best sampled point

The barrier distribution is the second energy derivative of $E\sigma(E)$,

$$
D(E)=\frac{d^2[E\sigma(E)]}{dE^2}\approx
\frac{F(E+\Delta E)-2F(E)+F(E-\Delta E)}{\Delta E^2},
\qquad F=E\sigma .
$$

`second_derivative` is copied from `34S+168Er/analysis/Barrier_distribution.ipynb`.
$F(E\pm\Delta E)$ is linearly interpolated when $E\pm\Delta E$ is not a measured
energy, and energies without a complete stencil are left undefined. The
experiment, CCFULL-ROM and exact Numerov curves use the same energies and
the same $\Delta E$. The experimental error is propagated from the mean of the
lower and upper cross-section uncertainties, assuming independent points.
Neighbouring $D(E)$ values share data points, so $\chi^2_D$ is only indicative.

An uncorrelated relative ROM error $\varepsilon$ in $\sigma$ changes $D(E)$ by
roughly $\sqrt6\,\varepsilon E\sigma/\Delta E^2$, so the ROM curve is reliable
only where it agrees with the exact Numerov curve.

In [6]:
# Barrier distribution; second_derivative is copied from
BARRIER_DELTA_E = 2 * (154.0 / (16.0 + 154.0))



def second_derivative(E, sigma, sigma_error, dE=3.33):
    """
    Return the fusion barrier distribution

        D(E) = d²[E * sigma(E)] / dE²

    from a centred three-point difference. When E ± dE is not one of the
    input energies, E*sigma is linearly interpolated.

    Parameters
    ----------
    E : array_like
        Centre-of-mass energies in MeV, strictly increasing.
    sigma : array_like
        Fusion cross sections.
    sigma_error : array_like
        Uncertainties of the fusion cross sections.
    dE : float
        Energy step of the second difference in MeV.

    Returns
    -------
    d2y_dx2 : ndarray
        d²(E*sigma)/dE².
    delta : ndarray
        Propagated uncertainty.

    Positions without a complete E-dE, E, E+dE stencil are returned as np.nan.
    """
    E = np.asarray(E, dtype=float)
    sigma = np.asarray(sigma, dtype=float)
    sigma_error = np.asarray(sigma_error, dtype=float)

    if not (E.shape == sigma.shape == sigma_error.shape):
        raise ValueError("E, sigma and sigma_error must have the same length")

    if E.ndim != 1:
        raise ValueError("The input data must be one-dimensional")

    if np.any(np.diff(E) <= 0):
        raise ValueError("E must be strictly increasing")

    if dE <= 0:
        raise ValueError("dE must be positive")

    # F(E) = E * sigma(E)
    F = E * sigma

    # Uncertainty of F: δF = E * δsigma
    F_error = E * sigma_error

    # Boundary points without a complete stencil are NaN rather than 0
    d2y_dx2 = np.full(E.shape, np.nan, dtype=float)
    delta = np.full(E.shape, np.nan, dtype=float)

    def add_interpolation_weights(coefficients, target_E, scale):
        """
        Add the linear-interpolation weights for target_E to coefficients.

        Explicit weights propagate the uncertainty correctly, including the
        case where one data point enters several interpolation terms.
        """
        right = np.searchsorted(E, target_E)

        # target_E is exactly one of the data points
        if (
            right < len(E)
            and np.isclose(E[right], target_E, rtol=0.0, atol=1.0e-10)
        ):
            coefficients[right] += scale
            return

        left = right - 1

        if left < 0 or right >= len(E):
            raise ValueError("Interpolation position is outside the data range")

        fraction = (
            (target_E - E[left])
            / (E[right] - E[left])
        )

        coefficients[left] += scale * (1.0 - fraction)
        coefficients[right] += scale * fraction

    for i, center_E in enumerate(E):
        lower_E = center_E - dE
        upper_E = center_E + dE

        # Keep NaN where E ± dE leaves the data range
        if lower_E < E[0] or upper_E > E[-1]:
            continue

        coefficients = np.zeros(len(E), dtype=float)

        # [F(E+dE) - 2F(E) + F(E-dE)] / dE²
        add_interpolation_weights(
            coefficients,
            lower_E,
            scale=1.0 / dE**2,
        )

        coefficients[i] += -2.0 / dE**2

        add_interpolation_weights(
            coefficients,
            upper_E,
            scale=1.0 / dE**2,
        )

        d2y_dx2[i] = coefficients @ F

        # The measured points are assumed to be independent
        delta[i] = np.sqrt(
            np.sum((coefficients * F_error) ** 2)
        )

    return d2y_dx2, delta


energies = experiment["energy_cm_mev"].to_numpy(dtype=float)
if not np.allclose(best_curve["energy_cm_mev"].to_numpy(dtype=float), energies):
    raise ValueError("The best-point curve and the experiment use different energies")
experimental_error = 0.5 * (
    experiment["experimental_uncertainty_lower_mb"].to_numpy(dtype=float)
    + experiment["experimental_uncertainty_upper_mb"].to_numpy(dtype=float)
)
barrier_experiment, barrier_experiment_error = second_derivative(
    energies,
    experiment["experimental_cross_section_mb"].to_numpy(dtype=float),
    experimental_error,
    dE=BARRIER_DELTA_E,
)
no_error = np.zeros_like(energies)
barrier_table = pd.DataFrame({
    "energy_cm_mev": energies,
    "experiment_mb_per_mev": barrier_experiment,
    "experiment_error_mb_per_mev": barrier_experiment_error,
    "rom_mb_per_mev": second_derivative(
        energies, best_curve["emulator_cross_section_mb"].to_numpy(dtype=float),
        no_error, dE=BARRIER_DELTA_E)[0],
})
if ("exact_numerov_cross_section_mb" in best_curve.columns
        and best_curve["exact_numerov_cross_section_mb"].notna().all()):
    barrier_table["exact_numerov_mb_per_mev"] = second_derivative(
        energies, best_curve["exact_numerov_cross_section_mb"].to_numpy(dtype=float),
        no_error, dE=BARRIER_DELTA_E)[0]

defined = barrier_table.dropna(subset=["experiment_mb_per_mev"])
defined = defined[defined["experiment_error_mb_per_mev"] > 0.0]
for column, label in (("rom_mb_per_mev", "CCFULL-ROM"),
                      ("exact_numerov_mb_per_mev", "Exact Numerov")):
    if column in defined.columns:
        pull = ((defined[column] - defined["experiment_mb_per_mev"])
                / defined["experiment_error_mb_per_mev"])
        peak = defined.loc[defined[column].idxmax(), "energy_cm_mev"]
        print(f"{label}: chi2_D/N = {np.mean(pull**2):.3g} over {len(defined)} "
              f"energies; maximum at {peak:.2f} MeV")
if "exact_numerov_mb_per_mev" in defined.columns:
    gap = (defined["rom_mb_per_mev"] - defined["exact_numerov_mb_per_mev"]).abs()
    print(f"|D_ROM - D_exact|: median {gap.median():.3g}, "
          f"maximum {gap.max():.3g} mb/MeV")
barrier_table

CCFULL-ROM: chi2_D/N = 7.42 over 33 energies; maximum at 60.57 MeV
Exact Numerov: chi2_D/N = 7.4 over 33 energies; maximum at 60.57 MeV
|D_ROM - D_exact|: median 0.234, maximum 0.728 mb/MeV


,energy_cm_mev,experiment_mb_per_mev,experiment_error_mb_per_mev,rom_mb_per_mev,exact_numerov_mb_per_mev
0,52.4144,NaN,NaN,NaN,NaN
1,52.8673,NaN,NaN,NaN,NaN
2,53.3202,NaN,NaN,NaN,NaN
3,53.7732,NaN,NaN,NaN,NaN
4,54.2261,NaN,NaN,NaN,NaN
5,54.6791,148.164808,13.203455,165.988990,166.110893
6,55.1320,178.381420,12.273901,188.603448,188.761293
7,55.5849,197.216170,12.422409,204.697191,204.906615
8,56.0379,226.286699,15.720055,217.344659,217.600033
9,56.4908,236.809477,19.999958,229.977773,230.284146


In [7]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=barrier_table["energy_cm_mev"],
    y=barrier_table["experiment_mb_per_mev"],
    error_y={
        "type": "data",
        "array": barrier_table["experiment_error_mb_per_mev"],
        "visible": True,
    },
    mode="markers",
    marker={"color": "black", "size": 7, "symbol": "circle-open"},
    name="Experiment",
))
fig.add_trace(go.Scatter(
    x=barrier_table["energy_cm_mev"],
    y=barrier_table["rom_mb_per_mev"],
    mode="lines+markers",
    line={"color": "#0072B2", "width": 2},
    marker={"size": 5},
    name="CCFULL-ROM",
))
if "exact_numerov_mb_per_mev" in barrier_table.columns:
    fig.add_trace(go.Scatter(
        x=barrier_table["energy_cm_mev"],
        y=barrier_table["exact_numerov_mb_per_mev"],
        mode="lines",
        line={"color": "#D55E00", "width": 2, "dash": "dash"},
        name="Exact Numerov",
    ))

fig.update_layout(
    title=(f"{reaction_system}: best sampled point {best_index}"
           f"<br><sup>barrier distribution, ΔE = {BARRIER_DELTA_E:g} MeV; "
           f"{parameter_text}</sup>"),
    xaxis_title="Centre-of-mass energy (MeV)",
    yaxis_title="d²(Eσ)/dE² (mb/MeV)",
    template="plotly_white",
    height=600,
    legend={"x": 0.02, "y": 0.98},
    xaxis=dict(range=[50, 75]),
    yaxis=dict(range=[-100, 750]),

)
fig.show()